# 🍽️ Food Image Classifier + Nutrition Lookup
CNN (MobileNetV2 Transfer Learning) + CSV Nutrition Data

In [ ]:
!pip install tensorflow pandas numpy matplotlib -q
import tensorflow as tf, pandas as pd, numpy as np, matplotlib.pyplot as plt
print('TF:', tf.__version__)

TF: 2.20.0


## 1. Extract Dataset

In [ ]:
import zipfile
with zipfile.ZipFile('dataset (6).zip', 'r') as z:
    z.extractall('dataset')
print('Done.')

Done.


## 2. Load Nutrition CSV & Build Lookup

The CSV (`final_matched_dataset.csv`) maps dish names → `matched_name` (which matches our image class folders). We build a lookup table: **class_name → avg nutrition per 100g**.

In [ ]:
import pandas as pd

df = pd.read_csv('final_matched_dataset (1).csv')   # upload this CSV to Colab
print('CSV shape:', df.shape)
print(df.head(3))

# Normalize matched_name to match image folder names (spaces → underscores, lowercase)
df['class_key'] = df['matched_name'].str.lower().str.strip().str.replace(' ', '_')

# Manual aliases for classes not directly in matched_name
ALIASES = {
    'bakery':        'bread',        # closest match
    'bread':         'rye bread',
    'breakfast':     'breakfast sausage links',
    'burger':        'hamburger',
    'cake':          'yellow cake',
    'cheese':        'cream cheese',
    'chips':         'baked chips',
    'chocolate':     'chocolate bar',
    'coffee':        'coffee creamer',
    'fish':          'white fish',
    'fried_chicken': 'chicken',
    'grape':         'grapes',
    'ice_cream':     'ice cream cake',
    'juice':         'orange juice',
    'lettuce':       'romaine lettuce',
    'milk':          'oat milk',
    'noodles':       'egg noodles cooked',
    'pizza':         'margherita pizza',
    'popcorn':       'popcorn air popped',
    'potato':        'white potato',
    'rice':          'spanish rice',
    'salad':         'greek salad',
    'shawarma':      'kebab',        # closest available
    'smoothie':      'fruit smoothie',
    'strawberry':    'strawberry',
    'sushi':         'tuna sushi maki',
    'watermelon':    'watermelon',
    'yogurt':        'greek yogurt',
}

# Build mean nutrition per class_key
nutrition_by_key = (
    df.groupby('class_key')[['calories','protein','fat','carb']]
      .mean().round(2)
      .to_dict(orient='index')
)

def get_nutrition(class_name):
    """Return nutrition dict for a given image class name."""
    key = class_name.lower().replace(' ', '_')
    if key in nutrition_by_key:
        return nutrition_by_key[key]
    # Try alias
    alias_key = ALIASES.get(key, '').lower().replace(' ', '_')
    if alias_key and alias_key in nutrition_by_key:
        return nutrition_by_key[alias_key]
    # Partial match fallback
    for k in nutrition_by_key:
        if key in k or k in key:
            return nutrition_by_key[k]
    return {'calories': None, 'protein': None, 'fat': None, 'carb': None}

# Quick test
for test_class in ['chicken', 'pizza', 'sushi', 'apple']:
    print(f'{test_class:15s} → {get_nutrition(test_class)}')

CSV shape: (13829, 9)
             name matched_name  \
0   chicken handi      chicken   
1   chicken mandi      chicken   
2  sticky chicken      chicken   

                                         ingredients category portion  \
0  Chicken, Onion, Tomatoes, Garlic, Ginger paste...     meat   100 g   
1  Chicken, Basmati Rice, Water, Onion, Garlic, G...     meat   100 g   
2  Chicken drumsticks, Soy Sauce, Honey, Olive Oi...     meat   100 g   

   calories  protein   fat  carb  
0     166.0     21.4  1.79   0.0  
1     166.0     21.4  1.79   0.0  
2     166.0     21.4  1.79   0.0  
chicken         → {'calories': 166.0, 'protein': 21.4, 'fat': 1.79, 'carb': 0.0}
pizza           → {'calories': 240.0, 'protein': 10.6, 'fat': 9.41, 'carb': 24.7}
sushi           → {'calories': 173.0, 'protein': 5.66, 'fat': 3.77, 'carb': 10.4}
apple           → {'calories': 64.0, 'protein': 0.0, 'fat': 0.0, 'carb': 11.7}


In [ ]:
import os
print(os.listdir('.'))

['.config', 'final_matched_dataset (1).csv', 'dataset', 'dataset (6).zip', 'sample_data']


In [ ]:
!ls -R dataset

dataset:
dataset

dataset/dataset:
apple	   carrot     falafel	     lettuce	potato	    sushi
bakery	   cheese     fish	     mango	quinoa	    sweet_potato
banana	   chicken    fried_chicken  milk	rice	    taco
beef	   chips      garlic	     noodles	salad	    tea
bread	   chocolate  grape	     oats	salmon	    tomato
breakfast  coffee     hummus	     onion	sandwich    tuna
broccoli   cookies    ice_cream      orange	shawarma    turkey
burger	   corn       juice	     pasta	shrimp	    watermelon
burrito    cream      kebab	     pineapple	smoothie    yogurt
butter	   cucumber   lamb	     pizza	spinach
cake	   duck       lasagna	     popcorn	strawberry

dataset/dataset/apple:
apple_0.jpg    apple_154.jpg  apple_208.jpg  apple_262.jpg  apple_46.jpg
apple_100.jpg  apple_155.jpg  apple_209.jpg  apple_263.jpg  apple_47.jpg
apple_101.jpg  apple_156.jpg  apple_20.jpg   apple_264.jpg  apple_48.jpg
apple_102.jpg  apple_157.jpg  apple_210.jpg  apple_265.jpg  apple_49.jpg
apple_103.jpg  apple_158.jpg  

## 3. Load Image Dataset

In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

IMG_SIZE   = (224, 224)   # MobileNetV2 native size
BATCH_SIZE = 32
DATA_DIR   = 'dataset/dataset'

train_ds = image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_ds = image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f'{num_classes} classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
# No caching before augmentation — keeps augmentation random each epoch
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

Found 18120 files belonging to 64 classes.
Using 14496 files for training.
Found 18120 files belonging to 64 classes.
Using 3624 files for validation.
64 classes: ['apple', 'bakery', 'banana', 'beef', 'bread', 'breakfast', 'broccoli', 'burger', 'burrito', 'butter', 'cake', 'carrot', 'cheese', 'chicken', 'chips', 'chocolate', 'coffee', 'cookies', 'corn', 'cream', 'cucumber', 'duck', 'falafel', 'fish', 'fried_chicken', 'garlic', 'grape', 'hummus', 'ice_cream', 'juice', 'kebab', 'lamb', 'lasagna', 'lettuce', 'mango', 'milk', 'noodles', 'oats', 'onion', 'orange', 'pasta', 'pineapple', 'pizza', 'popcorn', 'potato', 'quinoa', 'rice', 'salad', 'salmon', 'sandwich', 'shawarma', 'shrimp', 'smoothie', 'spinach', 'strawberry', 'sushi', 'sweet_potato', 'taco', 'tea', 'tomato', 'tuna', 'turkey', 'watermelon', 'yogurt']


## 4. Build CNN Model (MobileNetV2 Transfer Learning)

**Why MobileNetV2?**
- Pretrained on ImageNet → already knows edges, textures, shapes
- Achieves 60%+ on 64-class food datasets out of the box
- Custom CNN from scratch gets stuck at ~39% due to overfitting

**Architecture:**
1. Data augmentation (inside model, training-only)
2. MobileNetV2 backbone (frozen initially)
3. GlobalAveragePooling → BatchNorm → Dense(256) → Dropout → Dense(64 classes)

In [ ]:
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications import MobileNetV2

# -- Augmentation block (applied only during training) --
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augmentation')

# -- Pretrained backbone --
base = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base.trainable = False   # freeze for phase 1

# -- Full model --
inputs  = Input(shape=(224,224,3))
x       = augment(inputs)
x       = tf.keras.applications.mobilenet_v2.preprocess_input(x)  # single normalisation
x       = base(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)   # ← num_classes, NOT hardcoded 64

model = models.Model(inputs, outputs)
model.summary(line_length=80)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)        │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ augmentation (Sequential)         │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ true_divide (TrueDivide)          │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ subtract (Subtract)               │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224 (Functional) │ (None, 7, 7, 1280)       │     2,257,984 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ global_average_pooling2d          │ (None, 1280)             │             0 │
│ (GlobalAveragePooling2D)          │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ batch_normalization               │ (None, 1280)             │         5,120 │
│ (BatchNormalization)              │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 256)              │       327,936 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 256)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_1 (Dense)                   │ (None, 64)               │        16,448 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 2,607,488 (9.95 MB)

 Trainable params: 346,944 (1.32 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

## 5. Phase 1 — Train Head (Base Frozen, 15 epochs)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=15, callbacks=callbacks
)

Epoch 1/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 1064s 2s/step - accuracy: 0.3782 - loss: 2.6083 - val_accuracy: 0.5690 - val_loss: 1.6581 - learning_rate: 0.0010
Epoch 2/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 1097s 2s/step - accuracy: 0.5117 - loss: 1.8590 - val_accuracy: 0.5938 - val_loss: 1.5288 - learning_rate: 0.0010
Epoch 3/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 1049s 2s/step - accuracy: 0.5609 - loss: 1.6228 - val_accuracy: 0.6062 - val_loss: 1.4744 - learning_rate: 0.0010
Epoch 4/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 1140s 2s/step - accuracy: 0.5819 - loss: 1.5171 - val_accuracy: 0.6225 - val_loss: 1.4237 - learning_rate: 0.0010
Epoch 5/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 1041s 2s/step - accuracy: 0.6026 - loss: 1.4283 - val_accuracy: 0.6289 - val_loss: 1.4089 - learning_rate: 0.0010
Epoch 6/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 1045s 2s/step - accuracy: 0.6153 - loss: 1.3563 - val_accuracy: 0.6278 - val_loss: 1.4064 - learning_rate: 0.0010
Epoch 7/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 1035s 2s/step - accuracy: 0.6304 - l

## 6. Phase 2 — Fine-tune Top Layers (Low LR, 20 epochs)

In [ ]:
# Unfreeze top 40 layers of MobileNetV2
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=20, callbacks=callbacks
)

## 7. Evaluation

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f'\n Validation Accuracy: {acc*100:.1f}%')
print(f'   Validation Loss:     {loss:.4f}')

## 8. Training Curves

In [ ]:
def concat_hist(h1, h2, key):
    return h1.history[key] + h2.history[key]

acc_tr  = concat_hist(history1, history2, 'accuracy')
acc_val = concat_hist(history1, history2, 'val_accuracy')
loss_tr = concat_hist(history1, history2, 'loss')
loss_val= concat_hist(history1, history2, 'val_loss')
split   = len(history1.history['accuracy'])
ep      = range(len(acc_tr))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, tr, vl, title in zip(axes,
    [acc_tr, loss_tr], [acc_val, loss_val],
    ['Accuracy', 'Loss']):
    ax.plot(ep, tr,  label='Train')
    ax.plot(ep, vl,  label='Val')
    ax.axvline(split, color='gray', linestyle='--', label='Fine-tune start')
    ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## 9. Save Model

In [ ]:
model.save('food_classifier.keras')
print('Model saved as food_classifier.keras')

## 10. Predict Image → Class + Nutrition Info

This is where the **two files connect**: the CNN predicts the food class from the image, then we look up its nutrition from the CSV.

In [ ]:
import numpy as np
from PIL import Image as PILImage

def predict_with_nutrition(image_path, top_k=3):
    """
    Given an image path:
    1. CNN predicts the food class
    2. Nutrition lookup from CSV returns calories/protein/fat/carb
    """
    # -- Load & preprocess image --
    img = PILImage.open(image_path).convert('RGB').resize((224, 224))
    arr = np.array(img)[np.newaxis, ...].astype('float32')

    # -- Predict --
    preds  = model.predict(arr, verbose=0)[0]
    top_idx = preds.argsort()[-top_k:][::-1]

    print(f'\n📸 Image: {image_path}')
    print('─' * 50)

    for rank, idx in enumerate(top_idx, 1):
        cls   = class_names[idx]
        conf  = preds[idx] * 100
        nutr  = get_nutrition(cls)

        print(f'  #{rank} {cls.upper():20s}  ({conf:.1f}% confidence)')
        if nutr['calories'] is not None:
            print(f'      Nutrition per 100g:')
            print(f'        Calories : {nutr["calories"]:.1f} kcal')
            print(f'         Protein  : {nutr["protein"]:.1f} g')
            print(f'         Fat      : {nutr["fat"]:.1f} g')
            print(f'         Carbs    : {nutr["carb"]:.1f} g')
        else:
            print(f'       No nutrition data available for this class')
        print()

    # Return top prediction
    best_cls  = class_names[top_idx[0]]
    best_conf = preds[top_idx[0]] * 100
    best_nutr = get_nutrition(best_cls)
    return {'class': best_cls, 'confidence': best_conf, 'nutrition': best_nutr}

# ── Example: run on a few validation images ──────────────────
import os, random

# Pick random images from the validation set
random.seed(42)
sample_images = []
for cls in random.sample(class_names, 5):
    cls_dir = os.path.join(DATA_DIR, cls)
    files   = os.listdir(cls_dir)
    if files:
        sample_images.append((os.path.join(cls_dir, random.choice(files)), cls))

for img_path, true_cls in sample_images:
    print(f'[True label: {true_cls}]')
    result = predict_with_nutrition(img_path)
    correct = '✅' if result['class'] == true_cls else '❌'
    print(f'  → Prediction: {correct}')
    print()

## 11. Visual Prediction Grid with Nutrition

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

random.seed(7)
vis_samples = []
for cls in random.sample(class_names, 6):
    cls_dir = os.path.join(DATA_DIR, cls)
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if files:
        vis_samples.append((os.path.join(cls_dir, random.choice(files)), cls))

for ax, (img_path, true_cls) in zip(axes, vis_samples):
    img = PILImage.open(img_path).convert('RGB').resize((224,224))
    arr = np.array(img)[np.newaxis,...].astype('float32')
    preds = model.predict(arr, verbose=0)[0]
    pred_cls = class_names[preds.argmax()]
    conf     = preds.max() * 100
    nutr     = get_nutrition(pred_cls)

    ax.imshow(img)
    color = 'green' if pred_cls == true_cls else 'red'
    title = f'Pred: {pred_cls} ({conf:.0f}%)\nTrue: {true_cls}'
    if nutr['calories']:
        title += f'\n{nutr["calories"]:.0f} kcal | {nutr["protein"]:.1f}g P'
    ax.set_title(title, color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Food Classifier + Nutrition Lookup', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

chatbot

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import io, re
from PIL import Image as PILImage
import numpy as np

# ─────────────────────────────────────────────
# 🔹 STATE
# ─────────────────────────────────────────────
_state = {
    "best_cls":  None,
    "best_conf": None,
    "top_preds": None,
    "nutr":      None,
}

# ─────────────────────────────────────────────
# 🔹 NUTRITION LOOKUP (uses notebook's nutrition_by_key + ALIASES)
# ─────────────────────────────────────────────
def _lookup_nutr(cls):
    key = cls.lower().replace(' ', '_')
    if key in nutrition_by_key:
        return nutrition_by_key[key]
    # Try ALIASES
    alias = ALIASES.get(key, '')
    alias_key = alias.lower().replace(' ', '_')
    if alias_key and alias_key in nutrition_by_key:
        return nutrition_by_key[alias_key]
    # Partial match fallback
    for k in nutrition_by_key:
        if key in k or k in key:
            return nutrition_by_key[k]
    return {}


# ─────────────────────────────────────────────
# 🔹 IMAGE ANALYSIS
# ─────────────────────────────────────────────
def analyse_image(img_bytes, top_k=3):
    img = PILImage.open(io.BytesIO(img_bytes)).convert('RGB').resize((224, 224))
    arr = np.array(img)[np.newaxis, ...].astype('float32')
    preds   = model.predict(arr, verbose=0)[0]
    top_idx = preds.argsort()[-top_k:][::-1]
    top_preds = [(class_names[i], float(preds[i]) * 100) for i in top_idx]
    best_cls  = top_preds[0][0]
    best_conf = top_preds[0][1]
    nutr      = _lookup_nutr(best_cls)   # ✅ direct lookup
    return best_cls, best_conf, top_preds, nutr


# ─────────────────────────────────────────────
# 🔹 BOT LOGIC
# ─────────────────────────────────────────────
def build_bot_reply(user_text, best_cls, best_conf, top_preds, nutr, first=False):
    txt = user_text.lower().strip()

    # ── First message after image upload ──
    if first:
        tops = "\n".join(
            f"  {i+1}. {c.replace('_',' ').title()}  {p:.1f}%"
            for i, (c, p) in enumerate(top_preds)
        )
        nutr_text = ""
        if nutr and nutr.get('calories') is not None:
            nutr_text = (
                f"\n\n🍽 Nutrition per 100g:\n"
                f"  🔥 Calories : {nutr['calories']:.0f} kcal\n"
                f"  💪 Protein  : {nutr['protein']:.1f} g\n"
                f"  🧈 Fat      : {nutr['fat']:.1f} g\n"
                f"  🍞 Carbs    : {nutr['carb']:.1f} g"
            )
        return (
            f"📸 Image analysed!\n\n"
            f"🏆 Top predictions:\n{tops}\n\n"
            f"✅ Best guess: <b>{best_cls.replace('_',' ').title()}</b> ({best_conf:.1f}%)"
            f"{nutr_text}\n\n"
            f"💬 Ask me anything about this food!"
        )

    # ── No image uploaded yet — general food chat ──
    if best_cls is None:
        if any(w in txt for w in ['cal', 'calori', 'كالوري', 'سعر', 'حرار', 'kcal']):
            return "🔥 Calories depend on the food! Upload an image and I'll tell you exactly."
        if any(w in txt for w in ['protein', 'بروتين']):
            return "💪 Protein varies by food. Upload an image for specific info!"
        if any(w in txt for w in ['fat', 'دهن', 'دهون']):
            return "🧈 Fat content depends on the food. Try uploading an image!"
        if any(w in txt for w in ['carb', 'كارب', 'نشا', 'sugar', 'سكر']):
            return "🍞 Carbs vary a lot! Upload a food image and I'll check for you."
        if any(w in txt for w in ['healthy', 'health', 'رجيم', 'صح', 'مفيد']):
            return "⚖️ Generally: veggies & lean protein = healthy. Fried & sugary = less so! Upload an image for a specific answer."
        if any(w in txt for w in ['hi', 'hello', 'hey', 'مرحبا', 'أهلا', 'هاي']):
            return "👋 Hi! I'm your food assistant 🍽️ Upload a food image and I'll analyse it for you!"
        if any(w in txt for w in ['thanks', 'thank', 'شكرا', 'تسلم', 'يسلمو']):
            return "😊 You're welcome! Upload a food image anytime."
        if any(w in txt for w in ['pizza','burger','sushi','salad','chicken','rice',
                                   'بيتزا','برجر','سوشي','سلطة','دجاج','أرز']):
            return "😋 Sounds delicious! Upload a photo of it and I'll give you the full nutrition info."
        return "🤖 I'm your food assistant! Upload a food image 📷 and ask me about calories, protein, fat, carbs, or if it's healthy."

    # ── Calories ──
    if any(w in txt for w in ['cal', 'calori', 'كالوري', 'سعر', 'حرار', 'kcal']):
        if nutr and nutr.get('calories') is not None:
            return f"🔥 <b>{best_cls.replace('_',' ').title()}</b> has <b>{nutr['calories']:.0f} kcal</b> per 100g."
        return "No calorie data available for this food."

    # ── Protein ──
    if any(w in txt for w in ['protein', 'بروتين']):
        if nutr and nutr.get('protein') is not None:
            return f"💪 Protein: <b>{nutr['protein']:.1f}g</b> per 100g."
        return "No protein data available."

    # ── Fat ──
    if any(w in txt for w in ['fat', 'دهن', 'دهون']):
        if nutr and nutr.get('fat') is not None:
            return f"🧈 Fat: <b>{nutr['fat']:.1f}g</b> per 100g."
        return "No fat data available."

    # ── Carbs ──
    if any(w in txt for w in ['carb', 'كارب', 'نشا', 'sugar', 'سكر']):
        if nutr and nutr.get('carb') is not None:
            return f"🍞 Carbs: <b>{nutr['carb']:.1f}g</b> per 100g."
        return "No carbs data available."

    # ── Full nutrition ──
    if any(w in txt for w in ['nutrition', 'all', 'كل', 'كامل', 'summary', 'تغذي', 'قيمة']):
        if nutr and nutr.get('calories') is not None:
            return (
                f"📊 <b>{best_cls.replace('_',' ').title()}</b> per 100g:<br>"
                f"🔥 {nutr['calories']:.0f} kcal &nbsp;|&nbsp; "
                f"💪 {nutr['protein']:.1f}g protein &nbsp;|&nbsp; "
                f"🧈 {nutr['fat']:.1f}g fat &nbsp;|&nbsp; "
                f"🍞 {nutr['carb']:.1f}g carbs"
            )
        return "No nutrition data available."

    # ── Confidence ──
    if any(w in txt for w in ['confidence', 'confident', 'sure', 'نسبة', 'متأكد', 'مؤكد', 'percent']):
        return f"🎯 I'm <b>{best_conf:.1f}%</b> confident this is <b>{best_cls.replace('_',' ').title()}</b>."

    # ── Other predictions ──
    if any(w in txt for w in ['other', 'else', 'alternative', 'second', 'تاني', 'بديل', 'احتمال']):
        others = "<br>".join(
            f"  {i+2}. {c.replace('_',' ').title()} — {p:.1f}%"
            for i, (c, p) in enumerate(top_preds[1:])
        )
        return f"🔍 Other possibilities:<br>{others}"

    # ── Healthy ──
    if any(w in txt for w in ['healthy', 'health', 'good', 'bad', 'diet', 'رجيم', 'صح', 'مفيد']):
        if nutr and nutr.get('calories') is not None:
            cal = nutr['calories']
            tag = ("🟢 low-calorie — great choice!" if cal < 150
                   else "🟡 moderate — okay in portions." if cal < 300
                   else "🔴 high-calorie — enjoy in moderation.")
            return (
                f"⚖️ <b>{best_cls.replace('_',' ').title()}</b> is {tag}<br>"
                f"{cal:.0f} kcal | {nutr['protein']:.1f}g protein | "
                f"{nutr['fat']:.1f}g fat | {nutr['carb']:.1f}g carbs"
            )
        return "Not enough data to assess healthiness."

    # ── What is this ──
    if any(w in txt for w in ['what', 'ايه', 'إيه', 'identify', 'food', 'اكل', 'ده']):
        return f"🍽️ This looks like <b>{best_cls.replace('_',' ').title()}</b> ({best_conf:.1f}% confidence)."

    # ── Greetings ──
    if any(w in txt for w in ['hi', 'hello', 'hey', 'مرحبا', 'أهلا', 'هاي']):
        return f"👋 Hi! I detected <b>{best_cls.replace('_',' ').title()}</b>. What would you like to know?"

    # ── Thanks ──
    if any(w in txt for w in ['thanks', 'thank', 'شكرا', 'تسلم', 'يسلمو']):
        return "😊 You're welcome! Upload another image anytime."

    # ── Fallback ──
    return (
        f"🤔 I can answer about:<br>"
        f"• Calories / Protein / Fat / Carbs<br>"
        f"• Full nutrition summary<br>"
        f"• How confident I am<br>"
        f"• Other predictions<br>"
        f"• Is it healthy?<br><br>"
        f"(Detected: <b>{best_cls.replace('_',' ').title()}</b>)"
    )


# ─────────────────────────────────────────────
# 🔹 CHAT RENDER
# ─────────────────────────────────────────────
def show_msg(msg, who="bot"):
    bg    = "#e8f5e9" if who == "bot" else "#e3f2fd"
    align = "left"    if who == "bot" else "right"
    icon  = "🤖"       if who == "bot" else "🧑"
    msg_html = msg.replace('\n', '<br>')
    with chat_out:
        display(HTML(f"""
        <div style="text-align:{align}; margin:5px 2px;">
          <span style="font-size:1.05em">{icon}</span>
          <span style="
            display:inline-block; max-width:80%; padding:9px 13px;
            background:{bg}; border-radius:12px;
            font-size:0.91em; color:#111; text-align:left;
            font-family:'Segoe UI',sans-serif;
            box-shadow:0 1px 3px rgba(0,0,0,.07);
          ">{msg_html}</span>
        </div>
        """))


# ─────────────────────────────────────────────
# 🔹 WIDGETS
# ─────────────────────────────────────────────
upload_btn = widgets.FileUpload(
    accept='image/*', multiple=False,
    description='📷 Upload Image',
    layout=widgets.Layout(width='180px')
)

text_input = widgets.Text(
    placeholder='Ask something about food...',
    layout=widgets.Layout(width='72%', height='36px')
)

send_btn = widgets.Button(
    description='Send ➤', button_style='primary',
    layout=widgets.Layout(width='90px', height='36px')
)

clear_btn = widgets.Button(
    description='🗑 Clear', button_style='warning',
    layout=widgets.Layout(width='80px', height='36px')
)

chat_out = widgets.Output(layout=widgets.Layout(
    border='1px solid #ddd',
    min_height='300px', max_height='420px',
    overflow_y='auto', padding='10px',
))

img_out = widgets.Output(layout=widgets.Layout(
    width='200px', min_height='50px'
))

status_lbl = widgets.HTML(
    value='<span style="color:gray;font-size:.85em">No image uploaded yet.</span>'
)


# ─────────────────────────────────────────────
# 🔹 EVENTS
# ─────────────────────────────────────────────
def on_upload(change):
    if not upload_btn.value:
        return
    fname     = list(upload_btn.value.keys())[0]
    img_bytes = bytes(upload_btn.value[fname]['content'])

    with img_out:
        clear_output(wait=True)
        display(PILImage.open(io.BytesIO(img_bytes)).resize((190, 190)))

    status_lbl.value = f'<span style="color:green;font-size:.85em">✅ Loaded: {fname}</span>'

    best_cls, best_conf, top_preds, nutr = analyse_image(img_bytes)
    _state.update(best_cls=best_cls, best_conf=best_conf,
                  top_preds=top_preds, nutr=nutr)

    show_msg(build_bot_reply('', best_cls, best_conf, top_preds, nutr, first=True), 'bot')


def on_send(_=None):
    user_text = text_input.value.strip()
    if not user_text:
        return
    text_input.value = ''
    show_msg(user_text, 'user')
    reply = build_bot_reply(
        user_text,
        _state['best_cls'], _state['best_conf'],
        _state['top_preds'], _state['nutr']
    )
    show_msg(reply, 'bot')


def on_clear(_):
    with chat_out:
        clear_output()
    with img_out:
        clear_output()
    _state.update(best_cls=None, best_conf=None, top_preds=None, nutr=None)
    status_lbl.value = '<span style="color:gray;font-size:.85em">No image uploaded yet.</span>'
    show_msg("👋 Hi! I'm your food assistant 🍽️<br>Upload a food image 📷 or ask me a general food question!", "bot")


upload_btn.observe(on_upload, names='value')
send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)   # ✅ Enter key sends

# ── Greeting on load ──
show_msg("👋 Hi! I'm your food assistant 🍽️<br>Upload a food image 📷 or ask me a general food question!", "bot")


# ─────────────────────────────────────────────
# 🔹 LAYOUT
# ─────────────────────────────────────────────
title = widgets.HTML("""
<div style="
  background:linear-gradient(135deg,#1b5e20,#43a047);
  color:white; padding:13px 18px; border-radius:10px; margin-bottom:8px;
  font-family:'Segoe UI',sans-serif;
">
  <h2 style="margin:0;font-size:1.3em;">🍽️ Food Chatbot</h2>
  <p style="margin:3px 0 0;font-size:.8em;opacity:.9;">
    Ask general food questions or upload an image for nutrition analysis!
  </p>
</div>
""")

left  = widgets.VBox([chat_out,
                      widgets.HBox([text_input, send_btn, clear_btn])],
                     layout=widgets.Layout(width='65%'))

right = widgets.VBox([
    widgets.HTML('<b style="font-family:Segoe UI;font-size:.88em">📷 Image preview</b>'),
    img_out,
    upload_btn,
    status_lbl
], layout=widgets.Layout(width='33%', padding='0 0 0 12px'))

display(title, widgets.HBox([left, right]))

In [ ]:
import torch

torch.save(model.state_dict(), "food_model.pth")